# DIT5411 Traditional Chinese Handwriting Recognition (Colab, Resume-Safe)

This notebook is designed for Google Colab.

**Key feature:** every expensive step (dataset download, extraction, train/test split,
model checkpoints, training history) is stored on **Google Drive**. If Colab disconnects
or shuts down overnight, you can reopen this notebook the next day, run the cells again,
and training will automatically resume from the last saved checkpoint.

Repository URL is already configured to `https://github.com/HenryLok0/DIT5411_LokChunHang`.


## 1. Mount Google Drive

All persistent files live on Drive so they survive runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configure project root on Drive

`DIT5411_PROJECT_ROOT` tells the Python scripts to put data/models/results on Drive.

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/DIT5411'
os.environ['DIT5411_PROJECT_ROOT'] = PROJECT_ROOT
os.makedirs(PROJECT_ROOT, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Drive space for project:')
!df -h $PROJECT_ROOT

## 3. Clone / update project repository on Drive

Clones `https://github.com/HenryLok0/DIT5411_LokChunHang` onto Drive so it persists across sessions.


In [ ]:
REPO_URL = 'https://github.com/HenryLok0/DIT5411_LokChunHang.git'
REPO_NAME = 'DIT5411_LokChunHang'
REPO_DIR = os.path.join(PROJECT_ROOT, REPO_NAME)

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print('Repository already exists; pulling latest changes...')
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
print('Working in:', REPO_DIR)


## 4. Install dependencies

Run this every time you get a fresh runtime.

In [ ]:
!pip install -q -r requirements.txt


## 5. Download full 13,065-character dataset (only once)

This downloads the dataset source repo, concatenates the split zip files, and extracts
them to `data/raw/cleaned_data`. If the folder already exists on Drive, the step is skipped.

In [ ]:
raw_data = os.path.join(PROJECT_ROOT, REPO_NAME, 'data', 'raw', 'cleaned_data')
zip_path = os.path.join(PROJECT_ROOT, REPO_NAME, 'data', 'raw', 'all_data.zip')
source_repo = '/tmp/handwritting_data_all'

if os.path.exists(raw_data) and len(os.listdir(raw_data)) > 0:
    print('Dataset already extracted at', raw_data)
else:
    print('Downloading dataset source repo...')
    !rm -rf $source_repo
    !git clone https://github.com/chenkenanalytic/handwritting_data_all.git $source_repo

    print('Concatenating zip parts...')
    !mkdir -p data/raw
    !cat $source_repo/all_data.zip* > $zip_path

    print('Extracting zip (this may take a while)...')
    !unzip -O big5 $zip_path -d data/raw

print('Raw dataset ready.')

## 6. Prepare train/test split (only once)

Splits the first 40 samples per class into `data/train` and the rest into `data/test`.
Skipped if the split folders already exist and are non-empty.

In [ ]:
train_dir = os.path.join(PROJECT_ROOT, REPO_NAME, 'data', 'train')
test_dir = os.path.join(PROJECT_ROOT, REPO_NAME, 'data', 'test')

if (os.path.exists(train_dir) and len(os.listdir(train_dir)) > 0 and
    os.path.exists(test_dir) and len(os.listdir(test_dir)) > 0):
    print('Train/test split already exists. Skipping prepare_data.py.')
else:
    print('Preparing train/test split...')
    !python prepare_data.py

print('Train/test split ready.')

## 7. Helper function for safe training

Trains a model and automatically resumes if a checkpoint already exists.
The `epochs` argument is the **total** number of epochs you want (including epochs
already completed).

In [ ]:
import json

def train_model(model_name, total_epochs=5, batch_size=1024):
    checkpoint = f'models/best_model_{model_name}.keras'
    history_path = f'results/history_{model_name}.json'
    completed = 0

    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
            completed = len(history.get('loss', []))
        print(f'Model {model_name.upper()}: found {completed} completed epoch(s).')
    else:
        print(f'Model {model_name.upper()}: starting from scratch.')

    if completed >= total_epochs:
        print(f'Model {model_name.upper()} already trained for {completed} epoch(s). Skipping.')
        return

    print(f'Training Model {model_name.upper()} from epoch {completed} to {total_epochs}...')
    !python train.py --model $model_name --epochs $total_epochs --batch-size $batch_size --resume

    # Verify history was saved to Drive
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
        print(f'Model {model_name.upper()} now has {len(history.get("loss", []))} epoch(s) recorded.')
    else:
        print(f'Warning: history file {history_path} not found after training.')


## 8. Train Model A (deep CNN)

If Colab disconnects, run this cell again the next day. It will resume automatically.

In [ ]:
train_model('a', total_epochs=5, batch_size=1024)


## 9. Train Model B (medium CNN)

In [ ]:
train_model('b', total_epochs=5, batch_size=1024)


## 10. Train Model C (small CNN)

In [ ]:
train_model('c', total_epochs=5, batch_size=1024)


## 11. Evaluate and compare all models

In [ ]:
!python evaluate.py --compare


## 12. Plot comparison chart

In [ ]:
!python visualize_history.py --compare


## 13. Keep Colab alive (optional)

Free Colab disconnects after ~90 minutes of inactivity. Running this cell in a separate
tab or in the background can delay the idle timeout, but it does not remove the 12-hour
hard limit. For guaranteed uninterrupted training, use Colab Pro.

In [ ]:
from IPython.display import display, Javascript
import time

def keep_alive(interval_seconds=600):
    """Simulate light activity to delay Colab idle timeout."""
    while True:
        display(Javascript('''
            (function() {
                const buttons = document.querySelectorAll('button');
                for (const b of buttons) {
                    if (b.innerText && b.innerText.includes('Reconnect')) {
                        b.click();
                    }
                }
            })();
        '''))
        time.sleep(interval_seconds)

# Run in background; stop manually when training is done
keep_alive()
